In [0]:
# Add path to engine files
import sys
import importlib

sys.path.append("/Workspace/Shared/Capstone/Databricks")

import table_config
import bronze_engine

importlib.reload(table_config)
importlib.reload(bronze_engine)

from bronze_engine import BronzeEngine
# Handle batch parameter safely
try:
    batch_number = dbutils.widgets.get("batch_number")
except:
    dbutils.widgets.text("batch_number", "1")
    batch_number = "1"

if batch_number == "live":
    # Check if live/ actually has files before running
    try:
        live_files = dbutils.fs.ls("s3://capstone-ecomm-team8/live/")
        real_files = [f for f in live_files if f.name.endswith(".csv")]
        if not real_files:
            print("live/ folder is empty — nothing to ingest. Skipping.")
            dbutils.notebook.exit("SKIPPED: no live files")
    except Exception:
        print("live/ folder does not exist or is empty — skipping.")
        dbutils.notebook.exit("SKIPPED: no live files")

# Run engine
print(f"Starting Bronze ingestion for batch: {batch_number}")
BronzeEngine(spark).run(batch_number)